In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os
from PIL import Image


In [ ]:
import shutil
import os

# Existing dataset (already in Drive)
OLD_DATASET = "/content/drive/MyDrive/cat_species_dataset"

# New project structure
BASE_DIR = "/content/drive/MyDrive/Cat_Species_CNN_Lab"
NEW_DATASET = f"{BASE_DIR}/original_dataset/cat_species_dataset"

# Create folders
os.makedirs(f"{BASE_DIR}/original_dataset", exist_ok=True)

# Copy dataset
if not os.path.exists(NEW_DATASET):
    shutil.copytree(OLD_DATASET, NEW_DATASET)
    print("? Dataset copied into project folder")
else:
    print("?? Dataset already exists in project folder")


In [ ]:
AUGMENTED_DATASET = f"{BASE_DIR}/augmented_dataset"
MODEL_SAVE_PATH = f"{BASE_DIR}/models/custom_cnn_cat_species.pth"

os.makedirs(AUGMENTED_DATASET, exist_ok=True)
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)


// data agumentation


In [ ]:

augment_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3
    )
])

TARGET_IMAGES_PER_SPECIES = 100

for species in os.listdir(NEW_DATASET):
    src_folder = os.path.join(NEW_DATASET, species)
    dst_folder = os.path.join(AUGMENTED_DATASET, species)
    os.makedirs(dst_folder, exist_ok=True)

    original_images = [
        img for img in os.listdir(src_folder)
        if img.endswith(".png")
    ]

    # 1?? Save original images first
    for img_name in original_images:
        image = Image.open(os.path.join(src_folder, img_name)).convert("RGB")
        image.save(os.path.join(dst_folder, img_name))

    current_count = len(original_images)
    aug_index = 0

    # 2?? Generate augmented images until ~100 reached
    while current_count < TARGET_IMAGES_PER_SPECIES:
        img_name = original_images[aug_index % len(original_images)]
        image = Image.open(os.path.join(src_folder, img_name)).convert("RGB")

        aug_img = augment_transform(image)
        aug_img.save(
            os.path.join(dst_folder, f"aug_{current_count+1}.png")
        )

        current_count += 1
        aug_index += 1

    print(f"{species}: total images = {current_count}")


In [ ]:
# Transform for training CNN
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(
    root=AUGMENTED_DATASET,
    transform=transform
)

class_names = dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Total images:", len(dataset))


In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


In [ ]:
import torch.nn as nn

class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
model = CustomCNN(num_classes).to("cuda")
print(model)


In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from tqdm import tqdm


In [ ]:
epochs = 25

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs}",
        leave=True
    )

    for images, labels in progress_bar:
        images, labels = images.to("cuda"), labels.to("cuda")

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # update progress bar with loss
        progress_bar.set_postfix(loss=loss.item())

    print(f"? Epoch {epoch+1} completed | Avg Loss: {running_loss/len(train_loader):.4f}")


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to("cuda"), labels.to("cuda")

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

accuracy = (correct / total) * 100
print(f"? Final Validation Accuracy: {accuracy:.2f}%")


In [ ]:
import torch

torch.save(model.state_dict(), MODEL_SAVE_PATH)
print("? Trained model saved successfully in Google Drive")


In [ ]:
print("\n?? TRAINING SUMMARY")
print("----------------------------")
print("Total classes      :", num_classes)
print("Total images       :", len(dataset))
print("Final accuracy (%) :", round(accuracy, 2))
print("Model saved at     :", MODEL_SAVE_PATH)


In [ ]:
# ===== Upload Google Image & Predict =====

import torch
from google.colab import files
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# 1?? Upload image using Colab UI
uploaded = files.upload()

# 2?? Load trained model
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

# 3?? Define transform (same as training)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 4?? Get uploaded image
image_name = list(uploaded.keys())[0]
image_path = f"/content/{image_name}"

image = Image.open(image_path).convert("RGB")
image_tensor = test_transform(image).unsqueeze(0).to("cuda")

# 5?? Predict
with torch.no_grad():
    output = model(image_tensor)
    _, predicted = torch.max(output, 1)

predicted_class = class_names[predicted.item()]

# 6?? Show result
plt.imshow(image)
plt.axis("off")
plt.title(f"Predicted Species: {predicted_class}")
plt.show()

print("?? Model Prediction:", predicted_class)
